### Automotive

In [0]:
import json
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt

import pyspark.sql.functions as f

In [0]:
# Paths
PICTURE_VOLUME_PATH = "/Volumes/agentbricks/volumes/pictures"

chart_name = "personal_vehicles_registered_per_year"
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

chart_filename = f"{chart_name}_{timestamp}.png"
metadata_filename = f"{chart_name}_{timestamp}.json"

chart_path = f"{PICTURE_VOLUME_PATH}/{chart_filename}"
metadata_path = f"{PICTURE_VOLUME_PATH}/{metadata_filename}"

In [0]:
# Load data
df_car_fleet_cz = spark.table("agentbricks.sector_data_bronze.car_fleet_cz")

df_chart = (
    df_car_fleet_cz
    .select(
        f.col("year"),
        f.col("oa_registred_number").alias("personal_vehicles_registered")
    )
    .where(f.col("year").isNotNull())
    .where(f.col("personal_vehicles_registered").isNotNull())
    .orderBy("year")
)

display(df_chart)

In [0]:
# Convert to pandas for charting
pdf_chart = df_chart.toPandas()

In [0]:
# Create chart
plt.figure(figsize=(10, 6))

plt.plot(
    pdf_chart["year"],
    pdf_chart["personal_vehicles_registered"],
    marker="o",
)

plt.title("Personal Vehicles Registered in Czechia by Year")
plt.xlabel("Year")
plt.ylabel("Number of Registered Personal Vehicles")
plt.grid(True)
plt.tight_layout()

plt.savefig(chart_path, dpi=150)
plt.close()

print(f"Chart saved to: {chart_path}")

In [0]:
# Basic calculated insights for metadata
first_year = int(pdf_chart["year"].min())
last_year = int(pdf_chart["year"].max())

first_value = int(
    pdf_chart.loc[pdf_chart["year"] == first_year, "personal_vehicles_registered"].iloc[0]
)
last_value = int(
    pdf_chart.loc[pdf_chart["year"] == last_year, "personal_vehicles_registered"].iloc[0]
)

absolute_change = last_value - first_value
percentage_change = absolute_change / first_value * 100

In [0]:
# Metadata for Supervisor Agent / Genie
metadata = {
    "chart_id": chart_name,
    "chart_title": "Personal Vehicles Registered in Czechia by Year",
    "chart_filename": chart_filename,
    "chart_path": chart_path,
    "source_table": "agentbricks.sector_data_bronze.car_fleet_cz",
    "columns_used": ["year", "oa_registred_number"],
    "description": (
        "The chart shows the yearly development of registered personal vehicles "
        "in Czechia based on the oa_registred_number column."
    ),
    "suggested_markdown_reference": f"![Personal Vehicles Registered in Czechia by Year]({chart_path})",
    "suggested_commentary": (
        f"The number of registered personal vehicles increased from {first_value:,} in {first_year} "
        f"to {last_value:,} in {last_year}, representing an increase of "
        f"{absolute_change:,} vehicles, or approximately {percentage_change:.1f}%."
    ),
    "genie_instruction": (
        "Use the source table agentbricks.sector_data_bronze.car_fleet_cz to validate and enrich "
        "the interpretation of this chart. Refer specifically to year and oa_registred_number. "
        "Comment on the long-term trend, major changes, and what this implies for the automotive sector."
    ),
}

with open(metadata_path, "w", encoding="utf-8") as f_out:
    json.dump(metadata, f_out, indent=2, ensure_ascii=False)

print(f"Metadata saved to: {metadata_path}")

In [0]:
# Verify outputs
display(dbutils.fs.ls("dbfs:/Volumes/agentbricks/volumes/pictures"))